# Dataset export

<div style="text-align: justify">

Notebook used to export the LARD datasets into format usable by YOLO models. Make sure you have downloaded the datasets first, using the [`data-download.ipynb`](./data-download.ipynb) notebook.
</div>

> Notebook inspired from G. Delhomme's work. [[Github]](https://github.com/geoffrey-g-delhomme/lard-yolov8)

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
from pathlib import Path
from typing import (
    Union,
    Tuple,
    List,
)
import os
import shutil

import cv2
import json
import math
import numpy as np
import pandas as pd
import tqdm
import yaml

from swmf.monitors.ODD import comply_with_GLAC

In [56]:
SHOULD_VERIFY_GLAC = True

In [57]:
# List of data archives to read and export
train_archives = [
    ("LARD_train_BIRK_LFST.zip", "LARD_train_BIRK_LFST.csv"),
    ("LARD_train_DAAG_DIAP.zip", "LARD_train_DAAG_DIAP.csv"),
    ("LARD_train_KMSY.zip", "LARD_train_KMSY.csv"),
    ("LARD_train_LFMP_LFPO.zip", "LARD_train_LFMP_LFPO.csv"),
    ("LARD_train_LFQQ.zip", "LARD_train_LFQQ.csv"),
    ("LARD_train_LPPT_SRLI.zip", "LARD_train_LPPT_SRLI.csv"),
    ("LARD_train_VABB.zip", "LARD_train_VABB.csv"),
]
tests_archives = [
    ("LARD_test_synth.zip", "LARD_test_synth.csv"),
]

<div style="text-align: justify", class="alert alert-danger">

Make sure to have the correct path to the LARD datasets directory; the same one used in the [`data-download.ipynb`](./data-download.ipynb) notebook.
</div>

In [58]:
LARD_DATASETS_PATH = "../../LARD_dataset"

In [59]:
LARD_dpath = Path(LARD_DATASETS_PATH).resolve()
LARD_train_dpath = LARD_dpath / 'LARD_train'
LARD_tests_dpath = LARD_dpath / 'LARD_test'

## Generate images, labels and metadata

In [60]:
def convert_xyxy_to_xywh(
        bbox_xs: Union[np.ndarray, list],
        bbox_ys: Union[np.ndarray, list],
        img_w: int,
        img_h: int,
) -> np.ndarray: 
    """
    Convert the bounding box form xyxy (LARD format) to xywh (YOLO format).

    Note:
        The xyxy LARD bbox is in pixels while the xywh YOLO bbox is normalized to the image shape.

    Args:
        bbox_xs (Union[np.ndarray, list]): an array or list containing x's positions of the 4 bbox vertices.
        bbox_ys (Union[np.ndarray, list]): an array or list containing y's positions of the 4 bbox vertices.
        img_w (int): the original image's width
        img_h (int): the original image's height

    Returns:
        (np.ndarray) [4,] the YOLO-formated bbox.
    """
    xs = np.clip(bbox_xs, 0., img_w) / img_w  # Clip and normalize the box X coordinates
    ys = np.clip(bbox_ys, 0., img_h) / img_h  # Clip and normalize the box Y coordinates

    x_min = float(xs.min())
    x_max = float(xs.max())
    y_min = float(ys.min())
    y_max = float(ys.max())

    w  = x_max - x_min
    h  = y_max - y_min
    cx = x_min + w / 2.
    cy = y_min + h / 2.

    bbox = np.array([cx, cy, w, h])
    return bbox

<div style="text-align: justify">

For each image, we export the following (meta)data:
- image (HxW pixels, RGB)
- label (cx, cy, h, w, class)
- metadata:
    - The airport name
    - The runway ID (= airport name + runway name)
    - The date (`YYYYMMDD`)
    - The time (`hh:mm:ss`)
    - ATD (*along track distance*)
    - VPA (*vertical path angle*)
    - LPA (*lateral path angle*)
    - $\phi$ (*roll angle*)
    - $\theta$ (*pitch angle*)
    - $\psi$ (*yaw angle*)

The metadata are supposed to be available at runtime, coming from
- external metadata (time-of-day, airport ID, runway ID)
- *exteroreceptive* sensors (position = ATD, VPA, LAP),
- *introspective* sensors (attitude = $\phi, \theta, \psi $),

</div>

In [61]:
### HELPER ###
def get_runway_num(rn: str):
    """
    Util function to convert runway number into bearing angle
    """
    if not (isinstance(rn, int) or isinstance(rn, str)):
        raise TypeError("Input 'rn' must be either int or str.")
    
    if isinstance(rn, int):
        return rn
    else:
        s_type = type(rn)
        return int(s_type().join(filter(s_type.isdigit, rn)))
##############

In [1]:
### TEMP HOTFIX TO PROPERLY HANDLE YAW ANGLE ###
import pyproj
from skspatial.objects import Points

PATH_TO_RUNWAY_DATABASE = "../data/runways_database_all.json"


def ecef2llh(x, y, z):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/geo_utils.py
    """
    R = 6371010
    t = math.sqrt((x**2 + y**2 + z**2))

    if x == 0:
        if y < 0:
            lon = -math.pi / 2
        else:
            lon = math.pi / 2
    elif x < 0:
        if y < 0:
            lon = -math.pi + math.atan(y/x)
        else:
            lon = math.pi + math.atan(y/x)
    else:
        lon = math.atan(y/x)

    lat = math.asin(z/t)
    height = t - R
    return math.degrees(lat), math.degrees(lon), height 


def find_center(points):
    return Points(points).mean_center(return_centroid=True)


def find_azimuth_between_points(lon1, lat1, lon2, lat2):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/geo_utils.py#L258
    """
    return pyproj.Geod(ellps="WGS84").inv(lon1, lat1, lon2, lat2)


def get_runway_pts(database_file, airport_id, runway_id):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/ges_dataset.py#L86
    """
    with open(database_file, 'r') as f:
        runway_db = json.load(f)

    # Try adding 0 to the id...
    try:
        runway_pt = runway_db[airport_id][runway_id]
    except:
        runway_pt = runway_db[airport_id][f"{runway_id:0>2}"]

    _, ltp  = find_center([list(runway_pt['C']['position'].values()), list(runway_pt['D']['position'].values())])
    _, fpap = find_center([list(runway_pt['A']['position'].values()), list(runway_pt['B']['position'].values())])

    return runway_pt, ltp, fpap


def get_runway_yaw(database_file, airport_id, runway_id):
    """
    https://github.com/deel-ai/LARD/blob/LARD_V1/src/ges/ges_dataset.py#L107
    """
    _, ltp, fpap = get_runway_pts(database_file, airport_id, runway_id)

    fpap_lat, fpap_lon, _ = ecef2llh(*fpap)
    ltp_lat, ltp_lon, _ = ecef2llh(*ltp)
    rwy_psi = find_azimuth_between_points(ltp_lon, ltp_lat, fpap_lon, fpap_lat)
    
    print(airport_id, runway_id, '  \t', rwy_psi)
    return rwy_psi


def get_dist_between_angles(angle1, angle2):
    """
    Return the distance between two angles modulo 360
    """
    return min((angle1-angle2)%360, (angle2-angle1)%360)


def choose_runway_yaw(rwy_num, rwy_yaw_fwd, rwy_yaw_bwd):
    """
    Select the runway azimuth as close as the runway number as possible
    """
    if get_dist_between_angles(rwy_num*10, rwy_yaw_fwd) < get_dist_between_angles(rwy_num*10, rwy_yaw_bwd):
        return rwy_yaw_fwd
    else:
        return rwy_yaw_bwd

#####################################################

In [63]:
def export_one_sample(
        img_sample: Tuple[int, pd.Series],
        dst_dpath: Path,
        new_shape: Tuple[int, int],
        tasks: List[str],
) -> None:
    """
    Export one sample (image + label + metadata).

    Args:
        img_sample (Tuple[int, pd.Series]): the sample index + info.
        dst_dpath (Path): the directory path to save the sample into.
        new_shape (Tuple[int, int]): the new shape of exported sample.
        tasks (List[str]): the list of ML tasks to export the sample for.
    """
    im_indx, im_info = img_sample

    im_fpath = im_info["image_dpath"] / im_info["image"].replace('\\', '/')
    im = np.array(cv2.cvtColor(cv2.imread(im_fpath), cv2.COLOR_BGR2RGB))  # [H, W, C]
    h = im.shape[0]
    w = im.shape[1]

    # Crop watermark if necessary (top and bottom)
    watermark = im_info["watermark_height"]
    if not math.isnan(watermark):
        watermark = int(watermark)
        im = im[watermark: -watermark, :, :]

    # Size up the image and save it
    new_img_fpath = dst_dpath / 'images' / im_info['split'] / f"{im_indx:06d}.jpg"
    if not new_img_fpath.exists():
        im = cv2.resize(im, new_shape, interpolation=cv2.INTER_NEAREST)
        os.makedirs(new_img_fpath.parent, exist_ok=True)
        cv2.imwrite(new_img_fpath, cv2.cvtColor(im, cv2.COLOR_RGB2BGR))

    # Save the metadata (independent of the ML task)
    new_met_fpath = dst_dpath / "metadatas" / im_info['split'] / f"{new_img_fpath.stem}.txt"
    if not new_met_fpath.exists():
        os.makedirs(new_met_fpath.parent, exist_ok=True)
        with open(new_met_fpath, "w") as f:
            f.write(";".join(im_info.loc[['airport','rwy_id','date','time','ATD','VPA','LPA','phi','theta','psi']].astype(str).to_list()))

    # Compute labels
    x = np.array([im_info[f"x_{k}"] for k in "ABCD"], dtype=np.float32)
    y = np.array([im_info[f"y_{k}"] for k in "ABCD"], dtype=np.float32)

    if not math.isnan(watermark):
        y -= watermark
        h -= watermark * 2
    bbox = convert_xyxy_to_xywh(x, y, w, h)

    # For object detection
    if "detect" in tasks:
        tmp_image_fpath = dst_dpath / "task_detect" / "images" / im_info['split'] / new_img_fpath.name
        tmp_label_fpath = dst_dpath / "task_detect" / "labels" / im_info['split'] / f"{new_img_fpath.stem}.txt"
        tmp_mdata_fpath = dst_dpath / "task_detect" / "metadatas" / im_info['split'] / f"{new_img_fpath.stem}.txt"

        if not tmp_label_fpath.exists():
            os.makedirs(tmp_image_fpath.parent, exist_ok=True)
            os.makedirs(tmp_label_fpath.parent, exist_ok=True)
            os.makedirs(tmp_mdata_fpath.parent, exist_ok=True)
            # Handle image
            os.symlink(new_img_fpath, tmp_image_fpath, target_is_directory=False)
            # Handle metadata    
            os.symlink(new_met_fpath, tmp_mdata_fpath, target_is_directory=False)
            # Handle label
            with open(tmp_label_fpath, "w") as f:
                f.write("%g %.6f %.6f %.6f %.6f\n" % (0, *bbox))

    # For images segmentation
    if "segment" in tasks:
        kpts = np.stack((x/w, y/h), axis=-1).reshape(-1).tolist()  # [x1,x2] & [y1,y2] => [x1,y1,x2,y2]
        tmp_image_fpath = dst_dpath / "task_segment" / "images" / im_info['split'] / new_img_fpath.name
        tmp_label_fpath = dst_dpath / "task_segment" / "labels" / im_info['split'] / f"{new_img_fpath.stem}.txt"
        tmp_mdata_fpath = dst_dpath / "task_segment" / "metadatas" / im_info['split'] / f"{new_img_fpath.stem}.txt"

        if not tmp_label_fpath.exists():
            os.makedirs(tmp_image_fpath.parent, exist_ok=True)
            os.makedirs(tmp_label_fpath.parent, exist_ok=True)
            os.makedirs(tmp_mdata_fpath.parent, exist_ok=True)
            # Handle images symlink
            os.symlink(new_img_fpath, tmp_image_fpath, target_is_directory=False)
            # Handle metadata
            os.symlink(new_met_fpath, tmp_mdata_fpath, target_is_directory=False)
            # Handle labels
            with open(tmp_label_fpath, "w") as f:
                f.write("0 " + " ".join([f'{p:.6f}' for p in kpts]) + "\n")


def export_one_split_set(
        archives: List[Tuple[str, str]],
        dataset_split: str,
        src_dpath: Path,
        dst_dpath: Path,
        new_image_shape: Tuple[int, int],
        tasks: List[str],
) -> None:
    """
    Export each sample of the given dataset split (train or test).

    Args:
        archives
        dataset_name
        src_dpath
        dst_dpath
        new_image_shape
        tasks
    """
    # Get the csv filepath from unzipped archives
    csv_fpaths = [src_dpath / zip_fname.rpartition('.')[0] / csv_fname for zip_fname, csv_fname in archives]

    # Get the data from csv files
    dfs = []
    for csv_fpath in csv_fpaths:
        dfi = pd.read_csv(csv_fpath, delimiter=";")
        dfi["image_dpath"] = csv_fpath.parent
        dfi["split"] = dataset_split
        dfs.append(dfi)
    df = pd.concat(dfs).reset_index(drop=True).reset_index(drop=False)
    df['index'] = df['index'].map(lambda x: f"{x:06d}")

    #######################################################
    ### Preproc the dataFrames (TODO: put elsewhere...) ###    
    df.rename(columns={'time': 'datetime'}, inplace=True)
    df['watermark_height'] = df['watermark_height'].fillna(0.0)
    df['airport'] = df['airport'].astype(str)
    df['runway'] = df['runway'].astype(str)
    df['rwy_id'] = df['airport'] + '|' + df['runway'].map(lambda x: f"{x:0>3}")
    df['date'] = df['datetime'].map(lambda x: x.split(' ')[0])
    df['time'] = df['datetime'].map(lambda x: x.split(' ')[1])
    df['ATD'] = df['along_track_distance'] * 1852            # Convert from NM to m
    df['LPA'] = np.deg2rad(df['lateral_path_angle'])         # Convert from degrees to radians
    df['VPA'] = np.deg2rad(df['vertical_path_angle']) *(-1)  # Convert from degrees to radians and opposite
    df['slant_distance'] *= 1.852   # Convert from NM to km
    df['phi'] = np.deg2rad(df['roll'])

    ## Compute the runway azimuth ##
    df['rwy_yaw'] = 0.0
    for airport_id in df['airport'].unique():
        for runway_id in df[df['airport']==airport_id]['runway'].unique():
            rwy_num = get_runway_num(runway_id)
            rwy_yaw = get_runway_yaw(PATH_TO_RUNWAY_DATABASE, airport_id, runway_id)
            df.loc[(df['airport']==airport_id)&(df['runway']==runway_id), 'rwy_yaw'] = choose_runway_yaw(
                rwy_num, rwy_yaw[0], rwy_yaw[1]
            )

    df['psi'] = np.deg2rad((df['yaw'] - df['rwy_yaw'] + 180) % 360 - 180)
    df['theta'] = np.deg2rad(df['pitch'] - 90)
    df['X'] = -df['ATD']
    df['Y'] = +df['ATD'] * np.tan(df['LPA'])
    df['Z'] = +df['ATD'] * np.tan(df['VPA'])
    #######################################################
    #######################################################


    #######################################################
    #######################################################
    if SHOULD_VERIFY_GLAC and dataset_split=="train":
        samples_ODD_ok = comply_with_GLAC(df).astype(bool)
        df = df[samples_ODD_ok]
        df = df.drop(columns=['index'])
        df = df.reset_index(drop=True).reset_index(drop=False)
        df['index'] = df['index'].map(lambda x: f"{x:06d}")
    #######################################################
    #######################################################


    # Save csv file of exported metadata {img -> idx}
    os.makedirs(dst_dpath / "images", exist_ok=True)
    df.to_csv(dst_dpath / "images" / f"{dataset_split}_metadata.csv", sep=';', index=False, columns=["index", "image", "airport", "rwy_id"], header=False)

    # Export each sample individually
    for s in tqdm.tqdm(df.iterrows()):
        export_one_sample(s, dst_dpath, new_image_shape, tasks)


def launch_export(
        dataset_path: Path,
        dataset_name: str,
        imgsz: Tuple[int, int],
        tasks: List[str] = ["detect"],
        override_data: bool = False,
) -> Path:
    """
    Launch the export of the datasets. Give a name to it, new image shape et specific to ML task.

    Args:
        dataset_path (Path): the destination path to save the exported datasets.
        dataset_name (str): the name given to the exported datasets.
        imgsz (Tuple[int, int]): the image resolution to export.
        tasks (List[str]): the ML tasks for which to export the datasets.
        override_data (bool): flag to allow for data override in destination path.

    Returns:
        (Path) the path to exported datasets.
    """
    dataset_full_path = dataset_path / dataset_name

    def _make_yaml_file():
        for task in tasks:
            d = {
                'path': (dataset_full_path / f"task_{task}").as_posix(),
                'train': "images/train",
                'valid': "",
                'test': "images/test",
                'nc': 1,
                'names': {0: "runway"},
            }
            with open((dataset_full_path / f"task_{task}" / "data.yaml").as_posix(), "w") as f:
                yaml.dump(d, f, sort_keys=False)

    if dataset_full_path.exists():
        if not override_data:
            print(f"Destination path already exists ({dataset_full_path.as_posix()}). No action done.")
            return dataset_full_path
        else:
            shutil.rmtree(dataset_full_path)
    
    os.makedirs(dataset_full_path, exist_ok=True)

    print("Exporting train set.", flush=True, end=" ")
    export_one_split_set(train_archives, "train", LARD_train_dpath, dataset_full_path, imgsz, tasks)
    print("Done.", flush=True)

    print("Exporting test set.", flush=True, end=" ")
    export_one_split_set(tests_archives, "test" , LARD_tests_dpath, dataset_full_path, imgsz, tasks)
    print("Done.", flush=True)

    print("Creating dataset YAML file.", flush=True, end=" ")
    _make_yaml_file()
    print("Done.")

    return dataset_full_path

<div style="text-align: justify", class="alert alert-warning">

Launch the export of the LARD datasets by running the code below. (~20mn.)
</div>

In [64]:
# Initial dataset image 
LARD_EXPORT_RESOLUTION = (512, 512)  # W, H
LARD_EXPORT_PATH = "../data/datasets"
LARD_EXPORT_NAME = "lard_512x512_ICPR2026"    # Usually "lard_WxH"

In [65]:
lard_export_path = Path(LARD_EXPORT_PATH).resolve()

In [66]:
lard_path = launch_export(
    lard_export_path,
    LARD_EXPORT_NAME,
    imgsz=LARD_EXPORT_RESOLUTION,
    tasks=["detect", "segment"],
    override_data=True,
)

Exporting train set. BIRK 1   	 (175.2718380895036, -4.725904257654776, 1482.9439923135924)
BIRK 13   	 (-63.70754338074329, 116.27317768393974, 1163.8541115686448)
LFST 5   	 (-131.29428825419913, 48.687430756516335, 2397.960660988549)
LFST 23   	 (48.68743075651632, -131.29428825419913, 2397.960660988549)
DAAG 5   	 (-127.15648710402074, 52.82485791005844, 3499.9130263160337)
DAAG 23   	 (52.82485791005844, -127.15648710402074, 3499.9130263160337)
DIAP 3   	 (-157.55099961642142, 22.44805263967868, 3000.4640128272536)
DIAP 21   	 (22.448052639678686, -157.55099961642142, 3000.4640128272536)
KMSY 11   	 (-74.51765169348562, 105.46744070936957, 2985.806773638736)
KMSY 20   	 (15.474337298062528, -164.5227125600732, 2134.07645861)
KMSY 2   	 (-164.5227125600732, 15.474337298062522, 2134.07645861)
KMSY 29   	 (105.46744070936957, -74.51765169348562, 2985.806773638736)
LFMP 15   	 (-31.681497720750627, 148.308341573079, 2333.4513331833987)
LFPO 2   	 (-161.6555865331465, 18.33670393715743

7868it [05:22, 24.40it/s] 

Done.
Exporting test set. 

CYUL 06L   	 (42.670300855508756, -137.308963251519, 3355.0773030009386)
CYUL 24R   	 (-137.308963251519, 42.67030085550874, 3355.0773030009386)
CYVR 08L   	 (-80.12522492571156, 99.84601636184975, 2809.9051271654994)
CYVR 26R   	 (99.84601636184975, -80.12522492571156, 2809.9051271654994)
CYYZ 05   	 (46.48542058473445, -133.49470345837042, 3199.0238821575213)
CYYZ 23   	 (-133.49470345837042, 46.48542058473444, 3199.0238821575213)
DAAS 27   	 (-93.16865502385562, 86.81236949627225, 2896.25226025792)
DAAS 9   	 (86.81236949627225, -93.16865502385562, 2896.25226025792)
EDDV 09L   	 (92.56996168494213, -87.39274966302017, 3198.867351168784)
EDDV 27R   	 (92.57747340074889, -87.38524101227938, 3198.6231451979584)
EHAM 18R   	 (3.2082052101787975, -176.78949944668523, 3529.2680620587375)
EHAM 36L   	 (3.188524020489778, -176.80919412820674, 3530.1557708466203)
FMEP 15   	 (-50.97093357482637, 129.03455090826972, 2014.7862036860931)
FMEP 33   	 (129.03455090826972, -50.970933574826375, 201

2212it [01:28, 24.89it/s] 

Done.
Creating dataset YAML file. Done.


In [67]:
print(f"Path to exported LARD: {lard_path.as_posix()}")

Path to exported LARD: /home/dariom/Workspace/LARD_monitoring/data/datasets/lard_512x512_ICPR2026


## Generate train split strategies

<div style="text-align: justify">

This section concerns the definition of the strategies to split the train LARD dataset. Several options are explored:

1. **Basic random split**. This strategy is a simple train/valid split of ~80/20 proportion. Out of the 12212 train samples, let's take 10000 for actual train and 2212 for validation. No constraint is enforced on the split.

2. **Split per runway id**. In this strategy, we consider that images from the same runway are too similar and should not be in the train and validation sets in the same time (validation set would be useless). We thus split the train dataset per runway id. There are 27 unique runway ids and each runway record contains about 450 images. We take 22 runway ids for train (~10000 images) and 5 for validation (~2000 images).

3. **Split per airport id**. In this strategy, we consider that images from the same airport are too similar (approaching the same runway from both ends, or the same airport environment). We thus split the train dataset per airport id. There are 11 unique airport ids in the train set, containing from 450 to 1800 images. In order to respect the 80/20 proportion, we have to select between 2 and 5 airports for validation, leaving the remaining for training.
</div>

### Basic random split

<div style="text-align: justify" class="alert alert-danger">

Make sure to **run the entire section** when (re)generating the files of this split strategy, in order to always initiate the random seed properly and generate reproductible splits.
</div>

In [68]:
SEED = 42
np.random.seed(SEED)

In [69]:
# Define the number of training images
N_TRAIN = 6500 #10000

metadata_df = pd.read_csv(lard_path / "images" / "train_metadata.csv", sep=";", header=None, dtype=str, names=['index', 'image', 'airport', 'rwy_id'])

trainval_idx = np.random.permutation(metadata_df.shape[0])
train_idx = trainval_idx[:N_TRAIN]
valid_idx = trainval_idx[N_TRAIN:]

assert len(np.intersect1d(train_idx, valid_idx)) == 0, "[WARNING] Train/valid split indices are not disjoint."

metadata_df.loc[train_idx, "split"] = "train"
metadata_df.loc[valid_idx, "split"] = "valid"

metadata_df.to_csv(lard_path / "split_trainval.csv", columns=['index', 'split'], sep=';', index=False)
metadata_df['split'].value_counts()

split
train    6500
valid    1368
Name: count, dtype: int64

### Trainval split per airport

<div style="text-align: justify" class="alert alert-danger">

Make sure to **run the entire section** when (re)generating the files of this split strategy, in order to always initiate the random seed properly and generate reproductible splits.
</div>

<div style="text-align: justify">

In this section, we try and apply a split based on the airport id of each sample. 

TODO: code it
</div>

In [70]:
SEED = 42
np.random.seed(42)

In [71]:
AIRPORT_IDS = {
    'S': ['LFMP', 'LPPT'],
    'M': ['BIRK', 'DAAG', 'DIAP', 'LFST', 'SRLI'],
}
AIRPORT_SELECTS = {
    'S': 1,
    'M': 2,
}

metadata_df1 = pd.read_csv(lard_path / "images" / "train_metadata.csv", sep=";", header=None, dtype=str, names=['index', 'image', 'airport', 'rwy_id'])
valid_airports = np.concat([np.random.choice(AIRPORT_IDS[k], AIRPORT_SELECTS[k]) for k in AIRPORT_IDS.keys()])
metadata_df1['split'] = np.where(metadata_df1['airport'].isin(valid_airports), 'valid', 'train')
metadata_df1.to_csv(lard_path / "split_trainval_per_airport.csv", columns=["index", "split"], sep=';', index=False)

print("Selected VALID airports:")
print(valid_airports)
print(metadata_df1['split'].value_counts())

Selected VALID airports:
['LFMP' 'LFST' 'SRLI']
split
train    6427
valid    1441
Name: count, dtype: int64


### Trainval split per runway

<div style="text-align: justify" class="alert alert-danger">

Make sure to **run the entire section** when (re)generating the files of this split strategy, in order to always initiate the random seed properly and generate reproductible splits.
</div>

In [72]:
SEED = 42
np.random.seed(42)

In [73]:
metadata_df2 = pd.read_csv(lard_path / "images" / "train_metadata.csv", sep=";", header=None, dtype=str, names=['index', 'image', 'airport', 'rwy_id'])

trainval_runways = np.random.permutation(metadata_df2['rwy_id'].unique())
train_runways = trainval_runways[5:]  # Keep all but first 5 runways for training
valid_runways = trainval_runways[:5]  # Keep first 5 runways for validation

metadata_df2['split'] = np.where(metadata_df2['rwy_id'].isin(train_runways), "train", "valid")
metadata_df2.to_csv(lard_path / "split_trainval_per_runway.csv", columns=['index','split'], sep=';', index=False)

print("Selected VALID runways:")
print(valid_runways)
print(metadata_df2['split'].value_counts())

Selected VALID runways:
['KMSY|011' 'LFPO|002' 'KMSY|020' 'SRLI|014' 'BIRK|001']
split
train    6410
valid    1458
Name: count, dtype: int64
